# Example: CellCognition Time-Lapse Workflow

This notebook demonstrates a complete phenotyping workflow using the CellCognition dataset. It covers embedding extraction, property computation, clustering, and visualization.

**Dataset Information:**
- **Source:** [CellCognition Public Data](https://www.cellcognition.org/)
- **Description:** Segmented single-cell crops from time-lapse microscopy experiments.
- **Requirements:** Segmented single-cell crops with a `metadata.csv` file.

**Workflow Overview:**
1. Load metadata and map classes.
2. Extract deep learning embeddings using DINOv2.
3. Compute morphological and intensity properties.
4. Perform clustering and dimensionality reduction.
5. Explore results interactively.

In [ ]:
import torch
from phenome import PhenoMe, load_dinov2_model
import pandas as pd

# Set device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Initialize PhenoMe and load model
pheno = PhenoMe(device=device, seed=42)
wrapper = load_dinov2_model("dinov2_vitg14_reg", device=device)

## 1. Paths & Configuration

Specify the paths to your dataset and where to save checkpoints. By default, this notebook looks for data in `data/CellCognition/`.

In [ ]:
DATA_DIR = "/datasets/CellCognition/Chromatin_microtubules/preprocessed_H2b_20x_MD_exp911/images"
MASKS_DIR = "/datasets/CellCognition/Chromatin_microtubules/preprocessed_H2b_20x_MD_exp911/masks"
METADATA_PATH = "/datasets/CellCognition/Chromatin_microtubules/preprocessed_H2b_20x_MD_exp911/metadata.csv"

## 2. Load Metadata & Mapping

We load the metadata and apply a mapping to ensure classes are ordered logically (e.g., cell cycle progression).

In [ ]:
df_metadata = pd.read_csv(METADATA_PATH)
print(f"Loaded {len(df_metadata)} metadata entries.")

# Example mapping for H2B scheme (mitotic progression)
class_mapping = {
    "inter": "1_inter",
    "pro": "2_pro",
    "prometa": "3_prometa",
    "meta": "4_meta",
    "earlyana": "5_earlyana",
    "lateana": "6_lateana",
    "telo": "7_telo",
    "apo": "8_apo",
}
df_metadata["class"] = df_metadata["class"].map(class_mapping).fillna(df_metadata["class"])

# get only the rows in df_classes where the class is in get_lables
get_lables = ["1_inter", "2_pro", "3_prometa", "4_meta", "5_earlyana", "8_apo"]
df_metadata = df_metadata[df_metadata["class"].isin(get_lables)]

df_metadata.head()

In [ ]:
pheno.transfer_metadata_to_properties("rotation_angle")

## 3. Data Discovery

Discover image files and link them with the metadata.

In [ ]:
from phenome.utils import make_dataframe_metadata_fn

file_df = pheno.find_files(
    DATA_DIR,
    mask_dir=MASKS_DIR,
    metadata_fn=make_dataframe_metadata_fn(df_metadata)
)
file_df.head()

## 4. Feature Extraction

Extract deep learning embeddings (DINOv2) and compute classical morphological properties.

In [ ]:
# Process images to extract embeddings
pheno.process_images(
    wrapper,
    batch_size=128,
    num_workers=4,
    channel_mode="split",
    resize_size=182,  # Target size for resizing images
    pad_size=100,  # Pad small images to 224x224 before resizing to 224x224
)

# Compute classical properties (area, intensity, etc.)
df_properties = pheno.compute_properties(
    property_preset="basic",
    n_jobs=8,
)

## 5. Clustering & Dimensionality Reduction

Perform GMM clustering and visualize the phenospace using t-SNE.

In [ ]:
# Visualize with t-SNE
pheno.plot_centroids(
    group_by="class",
    method="tsne",
)

## 6. Interactive Exploration

Launch the interactive explorer to visually inspect the clusters and individual cells.

In [ ]:
e= pheno.create_interactive_explorer()